# Fleet Ranking with TOPSIS and Operational Risk Scoring

This notebook demonstrates:

1. **TopsisRanker** - multi-criteria fleet ranking with configurable weights
2. **OperationalRiskScorer** - multi-dimensional risk aggregation
3. Sensitivity analysis showing how weight changes affect rankings

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from anomalykit import TopsisRanker, OperationalRiskScorer
from anomalykit.ranking.topsis_ranker import AssetCriteriaInput
from generate_data import generate_fleet_kpis

## Data Generation

In [ ]:
df = generate_fleet_kpis(n=50, seed=42)
print(f"Fleet size: {len(df)} assets")
df.describe()

## TOPSIS Ranking - Default Weights

In [ ]:
criteria_cols = ["fuel_efficiency", "safety_score", "maintenance_cost", "uptime_pct", "emissions_index", "crew_satisfaction"]

benefit_criteria = {
    "fuel_efficiency": True,
    "safety_score": True,
    "maintenance_cost": False,   # lower is better
    "uptime_pct": True,
    "emissions_index": False,    # lower is better
    "crew_satisfaction": True,
}

assets = [
    AssetCriteriaInput(
        asset_id=row["asset_id"],
        criteria={c: row[c] for c in criteria_cols},
    )
    for _, row in df.iterrows()
]

ranker = TopsisRanker(benefit_criteria=benefit_criteria)
result_default = ranker.rank(assets)

print(f"Average score: {result_default.average_score:.4f}")
print(f"Quartiles: {result_default.quartile_boundaries}")
print(f"Underperformers: {len(result_default.underperformers)}")
print("\nTop 10:")
for r in result_default.rankings[:10]:
    print(f"  #{r.rank:2d} {r.asset_id}  score={r.overall_score:.4f}")

In [ ]:
top15 = result_default.rankings[:15]

fig, ax = plt.subplots(figsize=(12, 5))
ids = [r.asset_id for r in top15]
scores = [r.overall_score for r in top15]
colors = ["#4CAF50" if s >= result_default.quartile_boundaries["q3"] else
          "#FF9800" if s >= result_default.quartile_boundaries["q1"] else
          "#F44336" for s in scores]

ax.barh(ids[::-1], scores[::-1], color=colors[::-1], edgecolor="white")
ax.set_xlabel("TOPSIS Score")
ax.set_title("Fleet Ranking - Top 15 Assets (Default Equal Weights)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Sensitivity Analysis - Weight Configurations

Compare three weight profiles:
- **Equal**: all criteria weighted equally
- **Safety-first**: safety and compliance dominate
- **Cost-focused**: fuel efficiency and maintenance cost dominate

In [ ]:
weight_profiles = {
    "Equal": None,  # default equal weights
    "Safety-first": {
        "fuel_efficiency": 0.10,
        "safety_score": 0.35,
        "maintenance_cost": 0.10,
        "uptime_pct": 0.15,
        "emissions_index": 0.10,
        "crew_satisfaction": 0.20,
    },
    "Cost-focused": {
        "fuel_efficiency": 0.30,
        "safety_score": 0.10,
        "maintenance_cost": 0.30,
        "uptime_pct": 0.15,
        "emissions_index": 0.10,
        "crew_satisfaction": 0.05,
    },
}

rank_data = {}  # asset_id -> {profile: rank}

for profile_name, weights in weight_profiles.items():
    result = ranker.rank(assets, weights=weights)
    for r in result.rankings:
        rank_data.setdefault(r.asset_id, {})[profile_name] = r.rank

rank_df = pd.DataFrame(rank_data).T
rank_df.index.name = "asset_id"

# Show top 10 from each profile
print("Rank changes (top 15 by Equal weight):")
top_ids = [r.asset_id for r in result_default.rankings[:15]]
print(rank_df.loc[top_ids])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

show_ids = top_ids[:12]
x = np.arange(len(show_ids))
width = 0.25

for i, profile in enumerate(weight_profiles.keys()):
    ranks = [rank_df.loc[aid, profile] for aid in show_ids]
    ax.bar(x + i * width, ranks, width, label=profile, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(show_ids, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Rank (lower = better)")
ax.set_title("Ranking Sensitivity Analysis - How Weights Change Rankings")
ax.legend()
ax.invert_yaxis()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Operational Risk Scoring

In [ ]:
scorer = OperationalRiskScorer()

risk_results = []
for _, row in df.iterrows():
    r = scorer.score_asset(
        asset_id=row["asset_id"],
        asset_name=row["asset_name"],
        equipment_health_score=row["fuel_efficiency"],
        compliance_score=row["safety_score"],
        route_risk_score=row["emissions_index"],
    )
    risk_results.append(r)

risk_df = pd.DataFrame([
    {"asset_id": r.asset_id, "risk_score": r.overall_risk_score, "risk_level": r.risk_level}
    for r in risk_results
])

print("Risk level distribution:")
print(risk_df["risk_level"].value_counts())

### Risk Matrix Heatmap

In [ ]:
# Build a risk matrix from the first asset's result
sample_result = risk_results[0]
dim_names = [d.dimension.replace("_", " ").title() for d in sample_result.risk_dimensions]
dim_scores = [d.score for d in sample_result.risk_dimensions]
dim_weights = [d.weight for d in sample_result.risk_dimensions]

# Heatmap: dimensions across all assets
heat_data = []
for r in risk_results[:20]:  # top 20 assets
    heat_data.append([d.score for d in r.risk_dimensions])

heat_arr = np.array(heat_data)
heat_labels = [r.asset_id for r in risk_results[:20]]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heat_arr, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(dim_names)))
ax.set_xticklabels(dim_names, rotation=45, ha="right")
ax.set_yticks(range(len(heat_labels)))
ax.set_yticklabels(heat_labels, fontsize=8)
for i in range(heat_arr.shape[0]):
    for j in range(heat_arr.shape[1]):
        ax.text(j, i, f"{heat_arr[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.6, label="Risk Score")
ax.set_title("Operational Risk Heatmap - Risk Dimensions by Asset")
plt.tight_layout()
plt.show()

## Summary

- **TOPSIS** provides a transparent, reproducible ranking that respects both benefit and cost criteria
- **Weight sensitivity** reveals which assets are robust performers vs those whose rank depends heavily on the chosen priorities
- **OperationalRiskScorer** aggregates five risk dimensions into a single actionable score with risk matrix visualization